In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import re
import gc
import warnings
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os

In [ ]:
%rm -rf /kaggle/working/qwen_semeval

In [ ]:
base_path = '/kaggle/input/competitions/sem-eval-2026-task-13-subtask-a/Task_A'

files_to_explore = [
    'train.parquet',
    'validation.parquet',
    'test.parquet',
    'test_sample.parquet',
    'sample_submission.csv'
]

def explore_dataset_columns(path, files):
    for file_name in files:
        file_path = os.path.join(path, file_name)
        
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue
            
        print(f"File: {file_name}")
        
        try:
            # Load the dataframe based on the file extension
            if file_name.endswith('.parquet'):
                df = pd.read_parquet(file_path)
            elif file_name.endswith('.csv'):
                df = pd.read_csv(file_path)
            else:
                print("Unsupported file extension.")
                continue

            print(f"Total rows: {len(df):,}")
            print(f"Total columns: {len(df.columns)}")
            print("Columns & data types:")
            
            for col in df.columns:
                print(f" - {col}: {df[col].dtype}")

            print("\n")
            
        except Exception as e:
            print(f"Error reading {file_name}: {e}\n")


explore_dataset_columns(base_path, files_to_explore)

In [ ]:
# We are only analyzing files that contain the full feature set (labels, generators, languages)
files_to_analyze = [
    'train.parquet',
    'validation.parquet',
    'test_sample.parquet'
]

def analyze_distributions(path, files):
    for file_name in files:
        file_path = os.path.join(path, file_name)
        
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue
            
        print(f"========== Analyzing: {file_name} ==========")
        df = pd.read_parquet(file_path)
        
        # 1. Label Distribution
        print("\n--- Label Distribution ---")
        label_counts = df['label'].value_counts(dropna=False)
        label_pcts = df['label'].value_counts(normalize=True, dropna=False) * 100
        label_df = pd.DataFrame({'Count': label_counts, 'Percentage (%)': label_pcts}).round(2)
        print(label_df.to_string())
        
        # 2. Language Distribution
        print("\n--- Language Distribution ---")
        lang_counts = df['language'].value_counts(dropna=False)
        lang_pcts = df['language'].value_counts(normalize=True, dropna=False) * 100
        lang_df = pd.DataFrame({'Count': lang_counts, 'Percentage (%)': lang_pcts}).round(2)
        print(lang_df.to_string())
        
        # 3. Generator Distribution (Top 10)
        print("\n--- Generator Distribution (Top 10) ---")
        gen_counts = df['generator'].value_counts(dropna=False)
        gen_pcts = df['generator'].value_counts(normalize=True, dropna=False) * 100
        gen_df = pd.DataFrame({'Count': gen_counts, 'Percentage (%)': gen_pcts}).round(2)
        print(gen_df.head(10).to_string()) 
        if len(gen_counts) > 10:
            print(f"... and {len(gen_counts) - 10} more generators.")
            
        # 4. Basic Feature Extraction: Code Length
        print("\n--- Code Length Statistics (Character Count) ---")
        # Calculate length of code, handling potential nulls
        df['code_length'] = df['code'].fillna('').apply(len)
        print(df['code_length'].describe().round(2).to_string())
        
        print("\n" + "="*55 + "\n")


analyze_distributions(base_path, files_to_analyze)

In [ ]:
warnings.filterwarnings('ignore')
tqdm.pandas()

DATA_DIR = '/kaggle/input/competitions/sem-eval-2026-task-13-subtask-a/Task_A/'
DEBUG = False # Run on small sample subset to test code
N_FOLDS = 5

In [ ]:
from collections import Counter
class CodeFeatureExtractor:
    def __init__(self):
        self.camel_case = re.compile(r'[a-z]+[A-Z][a-z]+')
        self.snake_case = re.compile(r'[a-z]+_[a-z]+')
        self.operators = re.compile(r'[\+\-\*/%=\>\<]')
        self.logic_ops = re.compile(r'(\&\&|\|\||==|!=|>=|<=)')
        self.comments = re.compile(r'(//|#|/\*|"""|\'\'\')')
        self.words = re.compile(r'\b\w+\b')

    def _shannon_entropy(self, data):
        """Calculates Shannon entropy of a sequence"""
        if not data:
            return 0
        entropy = 0
        for x in Counter(data).values():
            p_x = float(x) / len(data)
            entropy -= p_x * math.log(p_x, 2)
        return entropy
    
    def extract(self, code):
        if not isinstance(code, str):
            code = ""
            
        features = {}
        lines = code.split('\n')
        total_chars = len(code) + 1 # +1 to avoid division by zero
        total_lines = len(lines) + 1
        
        # 1. Size & Structure
        features['line_count'] = total_lines
        features['char_count'] = total_chars
        
        features['avg_line_length'] = total_chars / total_lines
        line_lengths = [len(l) for l in lines]
        features['max_line_length'] = max(line_lengths) if line_lengths else 0
        features['line_length_std'] = np.std(line_lengths) if line_lengths else 0
        
        # 2. Whitespace & Formatting
        features['blank_line_ratio'] = sum(1 for l in lines if l.strip() == '') / total_lines
        features['trailing_whitespace_ratio'] = sum(1 for l in lines if l.endswith((' ', '\t')) and l.strip() != '') / total_lines
        
        # Indentation variance 
        indents = [len(l) - len(l.lstrip(' \t')) for l in lines if l.strip() != '']
        features['indentation_variance'] = np.var(indents) if indents else 0
        
        # 3. Casing & Naming
        features['camel_case_density'] = len(self.camel_case.findall(code)) / total_chars
        features['snake_case_density'] = len(self.snake_case.findall(code)) / total_chars
        
        # 4. Characters & Symbols
        alnum_count = sum(1 for c in code if c.isalnum())
        features['alphanumeric_ratio'] = alnum_count / total_chars
        
        # 5. Syntax & Logic Complexity
        features['operator_density'] = len(self.operators.findall(code)) / total_chars
        features['logic_density'] = len(self.logic_ops.findall(code)) / total_chars
        features['comment_density'] = len(self.comments.findall(code)) / total_lines
        
        # 6. Bracket Balance
        features['brace_balance'] = abs(code.count('{') - code.count('}'))
        features['paren_balance'] = abs(code.count('(') - code.count(')'))
        features['bracket_balance'] = abs(code.count('[') - code.count(']'))

        features['char_entropy'] = self._shannon_entropy(code)

        tokens = self.words.findall(code)
        features['token_entropy'] = self._shannon_entropy(tokens)
        
        return features

def process_dataframe(df):
    extractor = CodeFeatureExtractor()
    print(f"Extracting features for {len(df)} rows...")
    features_df = df['code'].progress_apply(lambda x: pd.Series(extractor.extract(x)))
    return pd.concat([df, features_df], axis=1)

In [ ]:
# Load Data
print("Loading data...")
train = pd.read_parquet(DATA_DIR + 'train.parquet')
val = pd.read_parquet(DATA_DIR + 'validation.parquet')
test = pd.read_parquet(DATA_DIR + 'test.parquet')
test_sample = pd.read_parquet(DATA_DIR + 'test_sample.parquet')

if DEBUG:
    train = train.sample(5000, random_state=42).reset_index(drop=True)
    val = val.sample(1000, random_state=42).reset_index(drop=True)
    test = test.sample(1000, random_state=42).reset_index(drop=True)
else:
    print("Undersampling training data to handle language and class imbalance...")
    MAX_SAMPLES = 12000 # C++ is 23k, so 12k/class roughly balances Python/C++/Java without losing too much
    
    # Check if language column exists to avoid errors on some variants
    if 'language' in train.columns and 'label' in train.columns:
        train = train.groupby(['language', 'label'], group_keys=False).apply(
            lambda x: x.sample(n=min(len(x), MAX_SAMPLES), random_state=42)
        )
        # Shuffle after groupby to ensure randomness
        train = train.sample(frac=1, random_state=42).reset_index(drop=True)
        print(f"Reduced training size to: {len(train)}")
        print(train['language'].value_counts())

# Extract Features
train = process_dataframe(train)
val = process_dataframe(val)
test = process_dataframe(test)
test_sample = process_dataframe(test_sample)

In [ ]:
# Define feature columns
feature_cols = [col for col in train.columns if col not in ['ID', 'code', 'generator', 'label', 'language']]
print(f"Extracted {len(feature_cols)} stylometric features.")

In [ ]:
final_features = feature_cols
print(final_features)

In [ ]:
def lgb_macro_f1(y_true, y_pred):
    y_pred_bin = (y_pred > 0.5).astype(int)
    f1 = f1_score(y_true, y_pred_bin, average='macro')
    return 'macro_f1', f1, True

X = train[final_features]
y = train['label']

X_val = val[final_features]
y_val = val['label']

# Parameters specifically chosen to prevent OOD memorization
lgb_params = {
    'objective': 'binary',
    'learning_rate': 0.05,
    'max_depth': 5,
    'num_leaves': 31,
    'colsample_bytree': 0.7, 
    'subsample': 0.8,
    'random_state': 42,
    'n_estimators': 500,
    'verbose': -1
}

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
val_preds = np.zeros(len(val))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

models = []
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold + 1} ---")
    X_train_f, y_train_f = X.iloc[train_idx], y.iloc[train_idx]
    X_valid_f, y_valid_f = X.iloc[valid_idx], y.iloc[valid_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    model.fit(
        X_train_f, y_train_f,
        eval_set=[(X_valid_f, y_valid_f)],
        eval_metric=lgb_macro_f1,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_preds[valid_idx] = model.predict_proba(X_valid_f)[:, 1]
    test_preds += model.predict_proba(test[final_features])[:, 1] / N_FOLDS
    val_preds += model.predict_proba(X_val[final_features])[:, 1] / N_FOLDS
    models.append(model)

print("\nTraining Complete.")

In [ ]:
print("--- Calibrating Threshold on Validation Set ---")

thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = []

for thresh in thresholds:
    # Use the validation set to find the best threshold
    preds_bin = (val_preds > thresh).astype(int)
    score = f1_score(y_val, preds_bin, average='macro')
    f1_scores.append(score)

best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Best Validation Macro F1: {best_f1:.4f} at Threshold: {best_thresh:.2f}")

# Plot threshold curve
plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, marker='o', markersize=3)
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best Thresh: {best_thresh:.2f}')
plt.title('Macro F1 vs. Decision Threshold')
plt.xlabel('Threshold')
plt.ylabel('Macro F1')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("--- Evaluating on OOD test_sample.parquet set ---")

FINAL_THRESHOLD=0.97

X_test_sample = test_sample[final_features]
y_test_sample = test_sample['label']

# Generate ensemble predictions
test_sample_preds = np.zeros(len(test_sample))
for model in models:
    test_sample_preds += model.predict_proba(X_test_sample)[:, 1] / N_FOLDS

# 1. Evaluate using the Standard Validation Threshold
preds_standard = (test_sample_preds > best_thresh).astype(int)
f1_standard = f1_score(y_test_sample, preds_standard, average='macro')

# 2. Evaluate using the OOD-Adjusted Threshold
preds_adjusted = (test_sample_preds > FINAL_THRESHOLD).astype(int)
f1_adjusted = f1_score(y_test_sample, preds_adjusted, average='macro')

print(f"Macro F1 (Standard Threshold = {best_thresh:.2f}): {f1_standard:.4f}")
print(f"Macro F1 (OOD-Adjusted Threshold = {FINAL_THRESHOLD:.2f}): {f1_adjusted:.4f}")
print("-" * 40)

best_eval_preds = preds_adjusted if f1_adjusted > f1_standard else preds_standard
best_eval_thresh = FINAL_THRESHOLD if f1_adjusted > f1_standard else best_thresh

print(f"\nChosen threshold: {best_eval_thresh:.2f}")
print(f"Accuracy: {accuracy_score(y_test_sample, best_eval_preds):.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test_sample, best_eval_preds)
cm_df = pd.DataFrame(cm, index=['Actual Human (0)', 'Actual AI (1)'], columns=['Predicted Human (0)', 'Predicted AI (1)'])
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test_sample, best_eval_preds, target_names=['Human (0)', 'AI (1)']))

In [ ]:
final_test_preds = (test_preds > FINAL_THRESHOLD).astype(int)

# Create submission file
submission = pd.DataFrame({
    'ID': test['ID'],
    'label': final_test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission saved successfully! Distribution of predictions:")
print(submission['label'].value_counts(normalize=True) * 100)

In [ ]:
import torch
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset

In [ ]:
# Block 2: Qwen-2.5-Coder Dataset Preparation
print("Preparing datasets for Qwen-2.5-Coder...")
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    # Implementing Head-Tail chunking for long sequences
    # 75% of character lengths are <= 1011, so 1024 tokens covers >90% of code perfectly.
    # For the few extremely long codes, we keep the first 512 and last 512 tokens to preserve structure.
    
    encoded = tokenizer(examples["code"], truncation=False)
    input_ids = encoded['input_ids']
    attention_mask = encoded['attention_mask']
    
    max_length = 1024
    
    new_input_ids = []
    new_attention_mask = []
    
    for ids, mask in zip(input_ids, attention_mask):
        if len(ids) > max_length:
            half = max_length // 2
            # Head and Tail
            ids = ids[:half] + ids[-half:]
            mask = mask[:half] + mask[-half:]
        else:
            # Pad if shorter
            padding_length = max_length - len(ids)
            ids = ids + [tokenizer.pad_token_id] * padding_length
            mask = mask + [0] * padding_length
            
        new_input_ids.append(ids)
        new_attention_mask.append(mask)

    return {"input_ids": new_input_ids, "attention_mask": new_attention_mask}

# Use the full undersampled 'train' set (~66k samples) rather than an arbitrary 12k subset.
# With Kaggle T4x2 (DDP) batch size 8 total, 66k samples takes roughly 8250 steps (approx 2.5 hours), completely safe from timeout.
train_qwen = train if not DEBUG else train.sample(100)

train_dataset = Dataset.from_pandas(train_qwen[['code', 'label']])
val_dataset = Dataset.from_pandas(val[['code', 'label']])
test_sample_dataset = Dataset.from_pandas(test_sample[['code', 'label']])

train_tokenized = train_dataset.map(preprocess_function, batched=True, remove_columns=['code'])
val_tokenized = val_dataset.map(preprocess_function, batched=True, remove_columns=['code'])
test_sample_tokenized = test_sample_dataset.map(preprocess_function, batched=True, remove_columns=['code'])

In [ ]:
%pip install -U bitsandbytes

In [ ]:
# Block 3: Qwen Model Setup & Training
print("Setting up QLoRA Model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)
# Enable gradient checkpointing to save memory on 1024 token sequences
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./qwen_semeval",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none",
    fp16=True,
    gradient_accumulation_steps=2
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    from sklearn.metrics import f1_score
    return {"macro_f1": f1_score(labels, predictions, average="macro")}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,  # Use full validation set
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None
if os.path.exists("./qwen_semeval"):
    last_checkpoint = get_last_checkpoint("./qwen_semeval")

if last_checkpoint is not None:
    print(f"Found existing weights! Resuming training or loading from {last_checkpoint}...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting Qwen Fine-tuning from scratch...")
    trainer.train()

In [ ]:
# Block 4: Qwen Validation & Test Inference
print("Running Qwen Inference...")

# We need probabilities for the whole val set and test_sample set
val_preds_qwen_raw = trainer.predict(val_tokenized)
test_sample_preds_qwen_raw = trainer.predict(test_sample_tokenized)

# Softmax to get probabilities
def get_probs(logits):
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

val_preds_qwen = get_probs(val_preds_qwen_raw.predictions)[:, 1]
test_sample_preds_qwen = get_probs(test_sample_preds_qwen_raw.predictions)[:, 1]

In [ ]:
# Evaluate Qwen independently on the test sample
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd

print("Qwen Standalone Evaluation on OOD test sample:")
qwen_preds_bin = (test_sample_preds_qwen > 0.6).astype(int)

qwen_f1 = f1_score(test_sample['label'], qwen_preds_bin, average='macro')
print(f"Macro F1 (Qwen Only): {qwen_f1:.4f}\n")

print("Qwen Confusion Matrix:")
cm_qwen = confusion_matrix(test_sample['label'], qwen_preds_bin)
print(pd.DataFrame(cm_qwen, index=['Actual Human (0)', 'Actual AI (1)'], columns=['Predicted Human (0)', 'Predicted AI (1)']))

print("\nClassification Report (Qwen Only):")
print(classification_report(test_sample['label'], qwen_preds_bin, target_names=['Human (0)', 'AI (1)']))


In [ ]:
# Block 5: The Cascade Meta-Classifier (Late-Fusion)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print("Training Meta-Classifier for Late-Fusion...")

X_meta_train = val[final_features].copy()
X_meta_train['lgbm_prob'] = val_preds
X_meta_train['qwen_prob'] = val_preds_qwen
y_meta_train = val['label']

scaler = StandardScaler()
X_meta_train_scaled = scaler.fit_transform(X_meta_train)

meta_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
meta_model.fit(X_meta_train_scaled, y_meta_train)

# For Test Sample
X_meta_test = test_sample[final_features].copy()
X_meta_test['lgbm_prob'] = test_sample_preds # from earlier LightGBM block
X_meta_test['qwen_prob'] = test_sample_preds_qwen
y_meta_test = test_sample['label']

X_meta_test_scaled = scaler.transform(X_meta_test)
meta_test_preds_prob = meta_model.predict_proba(X_meta_test_scaled)[:, 1]
meta_test_preds = (meta_test_preds_prob > 0.5).astype(int)

# Evaluation
f1_fusion = f1_score(y_meta_test, meta_test_preds, average='macro')
print(f"Macro F1 (Late-Fusion Cascade): {f1_fusion:.4f}")
print("Cascade Confusion Matrix:")
cm_fusion = confusion_matrix(y_meta_test, meta_test_preds)
cm_fusion_df = pd.DataFrame(cm_fusion, index=['Actual Human (0)', 'Actual AI (1)'], columns=['Predicted Human (0)', 'Predicted AI (1)'])
print(cm_fusion_df)
print("\nClassification Report (Cascade):")
print(classification_report(y_meta_test, meta_test_preds, target_names=['Human (0)', 'AI (1)']))